Nomes: Paula Martins, Eric Donato e Matheus Henrique

### Mapa

In [25]:
from typing import List, Dict

In [26]:
%%writefile entrada.txt
🙎 ⚪️ ⚪️ ⚪️ ⚪️ ⚪️
🟢 🧱 🧱 1️⃣ 8️⃣ ⚪️
⚪️ ⚪️ 🧱 ⚪️ ⚪️ ⚪️
⚪️ ⚪️ 🧱 ⚪️ ⚪️ ⚪️
⚪️ ⚪️ 🧱 ⚪️ ⚪️ ⚪️
⚪️ 🟢 ⚪️ ⚪️ ⚪️ ⚪️

Overwriting entrada.txt


In [27]:
VAZIO = '⚪️'
PAREDE = '🧱'
ALVO = '🟢'
AGENTE = '🙎'
PESOS = ['1️⃣','2️⃣','3️⃣','4️⃣','5️⃣','6️⃣','7️⃣','8️⃣','9️⃣']
dicionario_valor_caixas = {'1️⃣' : 1,  '2️⃣' : 2,  '3️⃣' : 3,  '4️⃣' : 4,  '5️⃣' : 5,  '6️⃣' : 6,  '7️⃣' : 7,  '8️⃣' : 8, '9️⃣' : 9}

In [28]:
def ler_grid(path: str) -> List[List[str]]:
    with open(path, "r", encoding="utf-8") as f:
        linhas = [ln.strip() for ln in f.readlines() if ln.strip()]
    grid = [ln.split() for ln in linhas]
    if not grid:
        raise ValueError("entrada.txt vazio.")
    m = len(grid[0])
    for i,row in enumerate(grid):
        if len(row) != m:
            raise ValueError(f"Grid não retangular: linha {i} tem {len(row)} colunas, esperado {m}")
    return grid

In [29]:
def parsear(grid: List[List[str]]):
    n = len(grid)
    m = len(grid[0])

    mapa = [VAZIO]*(n*m)
    walls = set()
    goals = []
    agent = None
    boxes_init: Dict[str,int] = {}

    for r in range(n):
        for c in range(m):
            tok = grid[r][c]
            pos = r*m + c

            # agente pode vir junto com outros símbolos (por segurança)
            if AGENTE in tok:
                agent = pos
                tok = tok.replace(AGENTE,"")

            if ALVO in tok:
                goals.append(pos)
                tok = tok.replace(ALVO,"")

            tok = tok.strip()

            if tok == PAREDE:
                walls.add(pos)
                mapa[pos] = PAREDE

            elif tok in PESOS:
                boxes_init[tok] = pos
                mapa[pos] = tok

            else:
                mapa[pos] = ALVO if pos in goals else VAZIO


    if agent is None:
        raise ValueError("Não encontrei 🙎 no mapa.")

    if len(goals) != len(boxes_init):
        raise ValueError(f"Deve haver 1 caixa para cada 🟢. Achei {len(boxes_init)} caixas e {len(goals)} alvos.")

    # ordem fixa das caixas (importante para estado hashável)
    box_ids = sorted(boxes_init.keys())

    return n, m, mapa, walls, goals, agent, boxes_init, box_ids


grid = ler_grid("entrada.txt")
n, m, mapa, walls, goals, agent0, boxes_init, box_ids = parsear(grid)

print("Tamanho:", n, "x", m)
print("Agente:", agent0)
print("Goals:", goals)
print("Boxes:", boxes_init)
print("Ordem caixas:", box_ids)

Tamanho: 6 x 6
Agente: 0
Goals: [6, 31]
Boxes: {'1️⃣': 9, '8️⃣': 10}
Ordem caixas: ['1️⃣', '8️⃣']


### Funções

In [30]:
def Dijkstra(custo_anterior, tem_caixa,qual_caixa):
  custo_anterior += 1
  if tem_caixa:
    return custo_anterior + dicionario_valor_caixas.get(qual_caixa,0)
  return custo_anterior

In [31]:
def Distancia_Manhattan(indice_A, indice_B):
    x_a = indice_A % m
    y_a = indice_A // m

    x_b = indice_B % m
    y_b = indice_B // m

    return float(abs(x_a - x_b) + abs(y_a - y_b))

In [32]:

def Calcular_Heuristica(no_atual):
    if len(no_atual.caixas_que_entreguei) == len(box_ids):
        return 0.0

    posicao_agente = no_atual.id
    heuristica_minima = float('inf')

    if no_atual.estou_com_caixa:
        for i in range(len(mapa)):
            if mapa[i] == '🟢' and i not in no_atual.caixas_que_entreguei.values():
                dist = Distancia_Manhattan(posicao_agente, i)
                if dist < heuristica_minima:
                    heuristica_minima = dist
        return heuristica_minima if heuristica_minima != float('inf') else 0.0
    else:
        entregues = no_atual.caixas_que_entreguei.keys()

        for caixa in box_ids:
            if caixa not in entregues:
                for pos, val in enumerate(mapa):
                    if val == caixa:
                        dist = Distancia_Manhattan(posicao_agente, pos)
                        if dist < heuristica_minima:
                            heuristica_minima = dist

        return heuristica_minima if heuristica_minima != float('inf') else 0.0

In [33]:
class No:
  def __init__(self, id,pai = None, movimento = None,estou_com_caixa = False,qual_caixa_estou = None,caixas_que_entreguei = None, custo = 0.0, heuristica = 0.0):
    self.id = id
    self.pai = pai
    self.movimento = movimento
    self.estou_com_caixa = estou_com_caixa
    self.qual_caixa_estou = qual_caixa_estou
    self.caixas_que_entreguei = caixas_que_entreguei if caixas_que_entreguei is not None else {}
    self.custo = custo
    self.heuristica = heuristica

  def Estado(self):

    estado = mapa.copy()

    if(self.estou_com_caixa):

      estado[self.id] = '🙎'+ '=' + self.qual_caixa_estou
      for i in range(len(mapa)):
        if(estado[i] == self.qual_caixa_estou):
          estado[i] = '⚪️'
    else:

      if(estado[self.id] == '⚪️'):
        estado[self.id] = '🙎'
      elif(estado[self.id] == '🟢'):
        estado[self.id] = '🙎'

      else:
        estado[self.id] = '🙎' + estado[self.id]
    for i in range(len(mapa)):
        if(estado[i] in self.caixas_que_entreguei):
          estado[i] = '⚪️'
    return estado

  def Imprimir_Grid(self):
        estado_atual = self.Estado()
        colunas = m

        for i in range(0, len(estado_atual), colunas):
            linha = estado_atual[i : i + colunas]

            # Converte cada item para string
            linha_formatada = [str(celula) for celula in linha]

            print(" \t│\t".join(linha_formatada))


  def Valor(self):

    if modo_busca == "estrela":
        return self.custo + self.heuristica

    elif modo_busca == "dijkstra":
        return self.custo

    elif modo_busca == "ganancioso":
        return self.heuristica

  def __lt__(self, outro):
    return self.Valor() < outro.Valor()


In [34]:
class Acao(No):

  def  sucessor_esquerda(self):
    posicao = self.id
    if (posicao % m != 0 and mapa[posicao-1] != '🧱'):
      proximo_custo = Dijkstra(self.custo,self.estou_com_caixa, self.qual_caixa_estou)
      filho = Acao(posicao-1, self, '⬅️', self.estou_com_caixa, self.qual_caixa_estou, self.caixas_que_entreguei, proximo_custo)
      filho.heuristica = Calcular_Heuristica(filho)
      return filho
    return None

  def sucessor_direita(self):
    posicao = self.id
    if (posicao % m != m-1 and mapa[posicao+1] != '🧱'):
      proximo_custo = Dijkstra(self.custo,self.estou_com_caixa, self.qual_caixa_estou)
      filho =  Acao(posicao+1, self, '➡️', self.estou_com_caixa, self.qual_caixa_estou, self.caixas_que_entreguei, proximo_custo)
      filho.heuristica = Calcular_Heuristica(filho)
      return filho
    return None

  def sucessor_cima(self):
    posicao = self.id
    if (posicao >= m and mapa[posicao-m] != '🧱'):
      proximo_custo = Dijkstra(self.custo,self.estou_com_caixa, self.qual_caixa_estou)
      filho = Acao(posicao-m, self, '⬆️', self.estou_com_caixa, self.qual_caixa_estou, self.caixas_que_entreguei, proximo_custo)
      filho.heuristica = Calcular_Heuristica(filho)
      return filho
    return None

  def sucessor_baixo(self):
    posicao = self.id
    if (posicao < len(mapa)-m and mapa[posicao+m] != '🧱'):
      proximo_custo = Dijkstra(self.custo,self.estou_com_caixa, self.qual_caixa_estou)
      filho = Acao(posicao+m, self, '⬇️', self.estou_com_caixa, self.qual_caixa_estou, self.caixas_que_entreguei, proximo_custo)
      filho.heuristica = Calcular_Heuristica(filho)
      return filho
    return None

  def pegar_caixa(self):
    posicao = self.id
    if (mapa[posicao] not in ('⚪️','🟢') and self.estou_com_caixa == False):
        proximo_custo = self.custo + 1
        filho = Acao(posicao,self,'🔴', True, mapa[posicao], self.caixas_que_entreguei, proximo_custo)
        filho.heuristica = Calcular_Heuristica(filho)
        return filho
    return None

  def soltar_caixa(self):
    posicao = self.id
    if (mapa[posicao] == '🟢' and self.estou_com_caixa == True):
      caixas_entregues = self.caixas_que_entreguei.copy()
      caixas_entregues[self.qual_caixa_estou] = posicao
      proximo_custo = self.custo + 1
      filho = Acao(posicao,self,'🟢', False, None, caixas_entregues, proximo_custo)
      filho.heuristica = Calcular_Heuristica(filho)
      return filho
    return None



In [35]:
class FilaDePrioridade:
    def __init__(self):
        self.heap = []

    def vazia(self):
        return len(self.heap) == 0

    def inserir(self, no):
        self.heap.append(no)
        self._subir(len(self.heap) - 1)

    def extrair_minimo(self):
        if self.vazia():
            return None

        if len(self.heap) == 1:
            return self.heap.pop()

        minimo = self.heap[0]

        self.heap[0] = self.heap.pop()

        self._descer(0)

        return minimo

    def _subir(self, indice):
        # Calcula o índice do pai usando divisão inteira
        indice_pai = (indice - 1) // 2

        # Enquanto não chegarmos à raiz e o nó atual for MENOR que o seu pai
        # (Nota: o operador '<' aqui vai invocar automaticamente o seu método mágico __lt__)
        if indice > 0 and self.heap[indice] < self.heap[indice_pai]:
            # Troca (Swap) de posições
            self.heap[indice], self.heap[indice_pai] = self.heap[indice_pai], self.heap[indice]

            # Continua a subir recursivamente
            self._subir(indice_pai)

    def _descer(self, indice):
        menor = indice
        filho_esquerda = 2 * indice + 1
        filho_direita = 2 * indice + 2
        tamanho = len(self.heap)

        # Verifica se o filho da esquerda existe e se é menor que o nó atual
        if filho_esquerda < tamanho and self.heap[filho_esquerda] < self.heap[menor]:
            menor = filho_esquerda

        # Verifica se o filho da direita existe e se é o menor de todos
        if filho_direita < tamanho and self.heap[filho_direita] < self.heap[menor]:
            menor = filho_direita

        # Se o menor não for o nó atual, ocorreu uma violação da regra do Min-Heap
        if menor != indice:
            # Fazemos o swap com o menor dos filhos
            self.heap[indice], self.heap[menor] = self.heap[menor], self.heap[indice]

            # Continua a descer recursivamente
            self._descer(menor)

In [36]:
class Problema:
  def __init__(self,id_inicial = 0):

    self.id_inicial = id_inicial

  def Objetivo(self,no_atual):
   return len(no_atual.caixas_que_entreguei) == len(boxes_init)

  def Mapa(self):
    return mapa



In [37]:
def caminho_feito (no):
  caminho = []
  while no is not None:
    caminho.append(no)
    no = no.pai
  caminho.reverse()
  return caminho

### Buscas

In [38]:
def busca(problema):

    if modo_busca == "estrela":
        print("A iniciar a Busca A*")

    elif modo_busca == "dijkstra":
        print("A iniciar a Busca Dijkstra")

    elif modo_busca == "ganancioso":
        print("A iniciar a Busca do Ganancioso")

    # 1. Instancia o nó inicial.
    # Como ele não nasce de uma "Acao", calculamos a heurística dele aqui uma única vez.
    no_inicial = Acao(id=problema.id_inicial, custo=0.0)
    no_inicial.heuristica = Calcular_Heuristica(no_inicial)

    # 2. Inicializa a Fila de Prioridade
    fronteira = FilaDePrioridade()
    fronteira.inserir(no_inicial)

    explorados = set()
    nos_expandidos = 0

    # 3. Loop principal
    while not fronteira.vazia():

        no_atual = fronteira.extrair_minimo()

        if no_atual is None:
            break
        # Verifica se atingiu o objetivo
        if problema.Objetivo(no_atual):
            print(f"Objetivo alcançado! Nós expandidos: {nos_expandidos}")
            print("Caminho feito:")
            for no_passado in caminho_feito(no_atual):
              print("movimento:" + str(no_passado.movimento) + " valor:" + str(no_passado.Valor()))
              no_passado.Imprimir_Grid()
              print()
            return no_atual

        chaves_entregues = tuple(sorted(no_atual.caixas_que_entreguei.keys()))
        assinatura_estado = f"{no_atual.id}-{no_atual.estou_com_caixa}-{no_atual.qual_caixa_estou}-{chaves_entregues}"

        if assinatura_estado in explorados:
            continue

        explorados.add(assinatura_estado)
        nos_expandidos += 1

        # 4. Gera os sucessores (agora eles já saem dessas funções com a heurística calculada!)
        sucessores = [
            no_atual.sucessor_esquerda(),
            no_atual.sucessor_direita(),
            no_atual.sucessor_cima(),
            no_atual.sucessor_baixo(),
            no_atual.pegar_caixa(),
            no_atual.soltar_caixa()
        ]

        # 5. Adiciona os filhos válidos na fronteira
        for filho in sucessores:
            if filho is not None:
                # O motor de busca ficou muito mais limpo. É só inserir!
                fronteira.inserir(filho)

    print("A busca terminou, mas nenhum caminho foi encontrado.")
    return None

In [39]:
modo_busca = "estrela"
resultado_estrela = busca(Problema(agent0))

A iniciar a Busca A*
Objetivo alcançado! Nós expandidos: 224
Caminho feito:
movimento:None valor:4.0
🙎 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️
🟢 	│	🧱 	│	🧱 	│	1️⃣ 	│	8️⃣ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	🟢 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️

movimento:➡️ valor:4.0
⚪️ 	│	🙎 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️
🟢 	│	🧱 	│	🧱 	│	1️⃣ 	│	8️⃣ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	🟢 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️

movimento:➡️ valor:4.0
⚪️ 	│	⚪️ 	│	🙎 	│	⚪️ 	│	⚪️ 	│	⚪️
🟢 	│	🧱 	│	🧱 	│	1️⃣ 	│	8️⃣ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	🟢 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️

movimento:➡️ valor:4.0
⚪️ 	│	⚪️ 	│	⚪️ 	│	🙎 	│	⚪️ 	│	⚪️
🟢 	│	🧱 	│	🧱 	│	1️⃣ 	│	8️⃣ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	🟢 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️

movimento:⬇️ valor:4.0
⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️
🟢 

In [40]:
modo_busca = "dijkstra"
resultado_dijkstra = busca(Problema(agent0))

A iniciar a Busca Dijkstra
Objetivo alcançado! Nós expandidos: 230
Caminho feito:
movimento:None valor:0.0
🙎 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️
🟢 	│	🧱 	│	🧱 	│	1️⃣ 	│	8️⃣ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	🟢 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️

movimento:➡️ valor:1.0
⚪️ 	│	🙎 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️
🟢 	│	🧱 	│	🧱 	│	1️⃣ 	│	8️⃣ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	🟢 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️

movimento:➡️ valor:2.0
⚪️ 	│	⚪️ 	│	🙎 	│	⚪️ 	│	⚪️ 	│	⚪️
🟢 	│	🧱 	│	🧱 	│	1️⃣ 	│	8️⃣ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	🟢 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️

movimento:➡️ valor:3.0
⚪️ 	│	⚪️ 	│	⚪️ 	│	🙎 	│	⚪️ 	│	⚪️
🟢 	│	🧱 	│	🧱 	│	1️⃣ 	│	8️⃣ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	🟢 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️

movimento:⬇️ valor:4.0
⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│

In [41]:
modo_busca = "ganancioso"
resultado_ganancioso = busca(Problema(agent0))

A iniciar a Busca do Ganancioso
Objetivo alcançado! Nós expandidos: 156
Caminho feito:
movimento:None valor:4.0
🙎 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️
🟢 	│	🧱 	│	🧱 	│	1️⃣ 	│	8️⃣ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	🟢 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️

movimento:➡️ valor:3.0
⚪️ 	│	🙎 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️
🟢 	│	🧱 	│	🧱 	│	1️⃣ 	│	8️⃣ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	🟢 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️

movimento:➡️ valor:2.0
⚪️ 	│	⚪️ 	│	🙎 	│	⚪️ 	│	⚪️ 	│	⚪️
🟢 	│	🧱 	│	🧱 	│	1️⃣ 	│	8️⃣ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	🟢 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️

movimento:➡️ valor:1.0
⚪️ 	│	⚪️ 	│	⚪️ 	│	🙎 	│	⚪️ 	│	⚪️
🟢 	│	🧱 	│	🧱 	│	1️⃣ 	│	8️⃣ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	⚪️ 	│	🧱 	│	⚪️ 	│	⚪️ 	│	⚪️
⚪️ 	│	🟢 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️

movimento:⬇️ valor:0.0
⚪️ 	│	⚪️ 	│	⚪️ 	│	⚪️ 	│	

### Print nos TXTs

In [42]:
def escrever_saida(nome_arquivo, no_final):

    with open(nome_arquivo, "w", encoding="utf-8") as f:

        if no_final is None:
            f.write("SEM SOLUCAO\n")
            return

        caminho = caminho_feito(no_final)
        ultimo = caminho[-1]

        # -------- Estado final --------
        f.write("Estado final\n")

        estado = ultimo.Estado()

        for i in range(0, len(estado), m):
            linha = estado[i:i+m]
            f.write(" ".join(linha) + "\n")

        # -------- Movimentos --------
        f.write("Movimentos\n")

        movimentos = ""
        for no in caminho:
            if no.movimento is not None:
                movimentos += no.movimento

        f.write(movimentos + "\n")

        # -------- Quantidade --------
        f.write("Quantidades de movimentos\n")
        f.write(str(len(movimentos)))
        print(movimentos)
        print(str(len(movimentos)))

In [43]:
print("Executando A*")
escrever_saida("a_estrela.txt", resultado_estrela)

print("Executando Dijkstra")
escrever_saida("dijkstra.txt", resultado_dijkstra)

print("Executando Busca Gananciosa")
escrever_saida("ganancioso.txt", resultado_ganancioso)

Executando A*
➡️➡️➡️⬇️➡️🔴⬆️⬅️⬅️⬅️⬅️⬇️🟢⬆️➡️➡️➡️⬇️🔴⬇️⬇️⬇️⬇️⬅️⬅️🟢
48
Executando Dijkstra
➡️➡️➡️⬇️🔴⬆️⬅️⬅️⬅️⬇️🟢⬆️➡️➡️➡️⬇️➡️🔴⬆️⬅️⬅️⬅️⬅️⬇️🟢
46
Executando Busca Gananciosa
➡️➡️➡️⬇️➡️🔴⬅️⬇️⬇️⬇️⬇️⬅️⬅️⬅️⬆️⬆️⬆️⬆️🟢⬆️➡️➡️➡️⬇️🔴⬇️⬇️⬇️⬇️⬅️⬅️🟢
60
